# VisionChat — Test Notebook

This notebook recreates the same COCO image-to-label mapping used during training, loads the saved `best_model.pth`, and evaluates the model on the held-out test split.

Because the project is multi-label, the notebook reports:
- test loss
- per-element accuracy
- exact-match accuracy
- classification report
- a confusion matrix for a selected class

> Update the paths below before running.

In [ ]:
import os
import re
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, multilabel_confusion_matrix

from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models
from torchvision.transforms import transforms


In [ ]:
# -----------------------------
# Paths and config
# -----------------------------
ANNOTATION_PATH = r"D:\instances_train2017.json\instances_train2017.json"
IMAGE_FOLDER = r"D:\Dataset\coco2017\train2017"
MODEL_PATH = "best_model.pth"

BATCH_SIZE = 32
TEST_SIZE = 0.15
RANDOM_STATE = 42
NUM_CLASSES = 80

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# -----------------------------
# Transform used during training
# -----------------------------
customTransform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [ ]:
# -----------------------------
# Load COCO annotations
# -----------------------------
with open(ANNOTATION_PATH, "r") as f:
    data = json.load(f)

print("Images in JSON:", len(data["images"]))
print("Annotations in JSON:", len(data["annotations"]))
print("Categories in JSON:", len(data["categories"]))


In [ ]:
# -----------------------------
# Build mapping exactly like training
# -----------------------------
num_images = len([name for name in os.listdir(IMAGE_FOLDER) if name.lower().endswith(".jpg")])
num_classes = len(data["categories"])

data["images"] = sorted(data["images"], key=lambda x: x["file_name"])
img_id_to_idx = {img["id"]: i for i, img in enumerate(data["images"])}

data["categories"] = sorted(data["categories"], key=lambda x: x["id"])
cat_id_to_idx = {cat["id"]: i for i, cat in enumerate(data["categories"])}

idx_to_cat_name = {i: cat["name"] for i, cat in enumerate(data["categories"])}

y = np.zeros((num_images, num_classes), dtype=int)

for ann in data["annotations"]:
    img_idx = img_id_to_idx[ann["image_id"]]
    cat_idx = cat_id_to_idx[ann["category_id"]]
    y[img_idx][cat_idx] = 1

print("Label matrix shape:", y.shape)


In [ ]:
# -----------------------------
# Custom dataset
# -----------------------------
class CustomDataset(Dataset):
    def __init__(self, folder, labels, transform):
        self.folder = folder
        files = [f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".png"))]
        self.files = sorted(files, key=lambda x: int(re.findall(r"\d+", x)[0]))
        self.labels = list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.folder, self.files[idx])
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return img, label


In [ ]:
# -----------------------------
# Build dataset and recreate split
# -----------------------------
dataset = CustomDataset(IMAGE_FOLDER, y, customTransform)

indices = np.arange(len(dataset))
trainVal_idx, test_idx = train_test_split(
    indices, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
train_idx, val_idx = train_test_split(
    trainVal_idx, test_size=0.15, random_state=RANDOM_STATE
)

test_dataset = Subset(dataset, test_idx)

print("Total samples:", len(dataset))
print("Test samples:", len(test_dataset))


In [ ]:
# -----------------------------
# DataLoader
# -----------------------------
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


In [ ]:
# -----------------------------
# Model definition must match training
# -----------------------------
vgg16 = models.vgg16(weights=None)

for param in vgg16.parameters():
    param.requires_grad = False

in_features = vgg16.classifier[0].in_features
vgg16.classifier = nn.Sequential(
    nn.Linear(in_features, 1024),
    nn.BatchNorm1d(1024),
    nn.ReLU(inplace=True),
    nn.Dropout(0.2),

    nn.Linear(1024, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(inplace=True),
    nn.Dropout(0.3),

    nn.Linear(512, NUM_CLASSES)
)

state_dict = torch.load(MODEL_PATH, map_location=device)
vgg16.load_state_dict(state_dict)
vgg16 = vgg16.to(device)
vgg16.eval()

print("Model loaded successfully.")


In [ ]:
# -----------------------------
# Test evaluation
# -----------------------------
criterion = nn.BCEWithLogitsLoss()

total_loss = 0.0
num_batches = 0

all_preds = []
all_labels = []

per_element_correct = 0
per_element_total = 0
exact_match_correct = 0
sample_total = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features = batch_features.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True).float()

        outputs = vgg16(batch_features)
        loss = criterion(outputs, batch_labels)

        total_loss += loss.item()
        num_batches += 1

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        per_element_correct += (preds == batch_labels).sum().item()
        per_element_total += batch_labels.numel()

        exact_match_correct += (preds.eq(batch_labels).all(dim=1)).sum().item()
        sample_total += batch_labels.size(0)

        all_preds.append(preds.cpu().numpy())
        all_labels.append(batch_labels.cpu().numpy())

all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)

avg_loss = total_loss / max(1, num_batches)
per_element_accuracy = 100.0 * per_element_correct / max(1, per_element_total)
exact_match_accuracy = 100.0 * exact_match_correct / max(1, sample_total)

print("\n==============================")
print(f"Test Loss: {avg_loss:.4f}")
print(f"Per-element Accuracy: {per_element_accuracy:.2f}%")
print(f"Exact-match Accuracy: {exact_match_accuracy:.2f}%")
print(f"Test Samples: {sample_total}")
print("==============================")


In [ ]:
# -----------------------------
# Classification report
# -----------------------------
print(classification_report(
    all_labels,
    all_preds,
    target_names=[idx_to_cat_name[i] for i in range(NUM_CLASSES)],
    zero_division=0
))


In [ ]:
# -----------------------------
# Confusion matrix for one selected class
# -----------------------------
selected_class_idx = 0   # change this to inspect another class
selected_class_name = idx_to_cat_name[selected_class_idx]

mcm = multilabel_confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = mcm[selected_class_idx].ravel()

cm = np.array([[tn, fp],
               [fn, tp]])

plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation="nearest")
plt.title(f"Confusion Matrix — {selected_class_name}")
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ["Pred 0", "Pred 1"])
plt.yticks(tick_marks, ["True 0", "True 1"])

threshold = cm.max() / 2.0
for i in range(2):
    for j in range(2):
        plt.text(j, i, format(cm[i, j], "d"),
                 ha="center",
                 va="center",
                 color="white" if cm[i, j] > threshold else "black")

plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# Optional: save predictions
# -----------------------------
np.save("all_preds.npy", all_preds)
np.save("all_labels.npy", all_labels)
print("Saved all_preds.npy and all_labels.npy")
